# Validation Notebook

Validates data integrity across Bronze, Silver, and Gold layers.
Results are persisted to `bda_gold.validation_results`.
Raises RuntimeError if critical checks fail.
Final section demonstrates that checks detect deliberately introduced errors.

In [ ]:
dbutils.widgets.dropdown("validation_scope", "all", ["gold", "all"])
validation_scope = dbutils.widgets.get("validation_scope")
# Portability: target catalog and source data location are parameters.
# Defaults reproduce the original setup (Databricks Free Edition); override them via widgets or job parameters.
import re
dbutils.widgets.text("catalog", "workspace", "Target catalog")
dbutils.widgets.text("source_schema", "samples.tpch", "Source data (catalog.schema)")
CATALOG = dbutils.widgets.get("catalog").strip()
SOURCE_SCHEMA = dbutils.widgets.get("source_schema").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG):
    raise ValueError(f"invalid catalog: {CATALOG!r}")
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*(\.[A-Za-z_][A-Za-z0-9_]*)?", SOURCE_SCHEMA):
    raise ValueError(f"invalid source_schema: {SOURCE_SCHEMA!r}")
spark.sql(f"USE CATALOG {CATALOG}")
print(f"catalog={CATALOG}  source_schema={SOURCE_SCHEMA}")

from pyspark.sql import Row
import uuid
from datetime import datetime

run_id = str(uuid.uuid4())
run_ts = datetime.now()
results = []

spark.sql("""
    CREATE TABLE IF NOT EXISTS bda_gold.validation_results (
        run_id STRING, checked_at TIMESTAMP, phase STRING, check_name STRING,
        layer STRING, expected_value STRING, actual_value STRING,
        passed BOOLEAN, is_critical BOOLEAN, details STRING
    ) USING DELTA
""")

def run_check(phase, check_name, layer, expected, query, is_critical=True):
    try:
        n = spark.sql(query).collect()
        actual = n[0][0] if n else None
        passed = (actual == expected) if expected is not None else (actual is not None)
        details = f"expected={expected}, actual={actual}"
    except Exception as e:
        actual, passed = None, False
        details = f"query error: {str(e)[:200]}"
    results.append(Row(run_id=run_id, checked_at=run_ts, phase=phase,
        check_name=check_name, layer=layer,
        expected_value=str(expected) if expected is not None else "null",
        actual_value=str(actual) if actual is not None else "null",
        passed=passed, is_critical=is_critical, details=details))
    print(f"  [{'PASS' if passed else 'FAIL'}] {check_name}: {details}")
    return passed

def chk0(phase, name, layer, query, is_critical=True):
    return run_check(phase, name, layer, 0, query, is_critical)

print(f"run_id={run_id}  checked_at={run_ts}")

In [ ]:
TPCH = ["customer", "lineitem", "nation", "orders", "part", "partsupp", "region", "supplier"]
phase = "bronze"
print(f"=== {phase.upper()} CHECKS ===")

for t in TPCH:
    src = spark.sql(f"SELECT COUNT(*) FROM {SOURCE_SCHEMA}.{t}").collect()[0][0]
    run_check(phase, f"row_count:{t}", "Bronze", src,
        f"SELECT COUNT(*) FROM bda_bronze.{t}")

# data equality both directions (EXCEPT ALL) and schema comparison
for t in TPCH:
    cols = [r.col_name for r in spark.sql(f"DESCRIBE {SOURCE_SCHEMA}.{t}").collect()
            if r.col_name and not r.col_name.startswith("#")]
    cl = ", ".join(cols)
    chk0(phase, f"data_eq_src2bronze:{t}", "Bronze",
        f"SELECT COUNT(*) FROM (SELECT {cl} FROM {SOURCE_SCHEMA}.{t} EXCEPT ALL SELECT {cl} FROM bda_bronze.{t})")
    chk0(phase, f"data_eq_bronze2src:{t}", "Bronze",
        f"SELECT COUNT(*) FROM (SELECT {cl} FROM bda_bronze.{t} EXCEPT ALL SELECT {cl} FROM {SOURCE_SCHEMA}.{t})")
    # schema comparison (Python-based, DESCRIBE cannot be used as SQL subquery)
    src_schema = set((r.col_name, r.data_type) for r in spark.sql(f"DESCRIBE {SOURCE_SCHEMA}.{t}").collect()
                     if r.col_name and not r.col_name.startswith("#"))
    bronze_schema = set((r.col_name, r.data_type) for r in spark.sql(f"DESCRIBE bda_bronze.{t}").collect()
                        if r.col_name and not r.col_name.startswith("#") and r.col_name != "ingestion_timestamp")
    schema_diff = len(src_schema.symmetric_difference(bronze_schema))
    results.append(Row(run_id=run_id, checked_at=run_ts, phase=phase,
        check_name=f"schema_eq:{t}", layer="Bronze",
        expected_value="0", actual_value=str(schema_diff),
        passed=(schema_diff == 0), is_critical=True,
        details=f"expected=0, actual={schema_diff}"))
    print(f"  [{'PASS' if schema_diff == 0 else 'FAIL'}] schema_eq:{t}: expected=0, actual={schema_diff}")
print()

In [ ]:
phase = "silver"
S = "bda_silver"
print(f"=== {phase.upper()} CHECKS ===")

chk0(phase, "part:null_brand", "Silver", f"SELECT COUNT(*) FROM {S}.part WHERE p_brand IS NULL OR TRIM(p_brand)=''")
chk0(phase, "part:null_mfgr", "Silver", f"SELECT COUNT(*) FROM {S}.part WHERE p_mfgr IS NULL OR TRIM(p_mfgr)=''")
chk0(phase, "part:dup_p_partkey", "Silver", f"SELECT COUNT(*) FROM (SELECT p_partkey FROM {S}.part GROUP BY p_partkey HAVING COUNT(*)>1)")
chk0(phase, "orders:dup_o_orderkey", "Silver", f"SELECT COUNT(*) FROM (SELECT o_orderkey FROM {S}.orders GROUP BY o_orderkey HAVING COUNT(*)>1)")
chk0(phase, "partsupp:null_keys", "Silver", f"SELECT COUNT(*) FROM {S}.partsupp WHERE ps_partkey IS NULL OR ps_suppkey IS NULL")
chk0(phase, "partsupp:dup_pairs", "Silver", f"SELECT COUNT(*) FROM (SELECT ps_partkey, ps_suppkey FROM {S}.partsupp GROUP BY ps_partkey, ps_suppkey HAVING COUNT(*)>1)")
chk0(phase, "lineitem:null_keys", "Silver", f"SELECT COUNT(*) FROM {S}.lineitem WHERE l_orderkey IS NULL OR l_linenumber IS NULL")
chk0(phase, "lineitem:dup_orderline", "Silver", f"SELECT COUNT(*) FROM (SELECT l_orderkey, l_linenumber FROM {S}.lineitem GROUP BY l_orderkey, l_linenumber HAVING COUNT(*)>1)")
chk0(phase, "lineitem->orders:orphans", "Silver", f"SELECT COUNT(*) FROM {S}.lineitem l LEFT ANTI JOIN {S}.orders o ON l.l_orderkey=o.o_orderkey")
chk0(phase, "lineitem->part:orphans", "Silver", f"SELECT COUNT(*) FROM {S}.lineitem l LEFT ANTI JOIN {S}.part p ON l.l_partkey=p.p_partkey")
chk0(phase, "lineitem->partsupp:orphans", "Silver", f"SELECT COUNT(*) FROM {S}.lineitem l LEFT ANTI JOIN {S}.partsupp ps ON l.l_partkey=ps.ps_partkey AND l.l_suppkey=ps.ps_suppkey")
chk0(phase, "partsupp->part:orphans", "Silver", f"SELECT COUNT(*) FROM {S}.partsupp ps LEFT ANTI JOIN {S}.part p ON ps.ps_partkey=p.p_partkey")
chk0(phase, "partsupp->supplier:orphans", "Silver", f"SELECT COUNT(*) FROM {S}.partsupp ps LEFT ANTI JOIN {S}.supplier s ON ps.ps_suppkey=s.s_suppkey")
chk0(phase, "part:nonpos_retailprice", "Silver", f"SELECT COUNT(*) FROM {S}.part WHERE p_retailprice IS NULL OR p_retailprice<=0")
chk0(phase, "lineitem:nonpos_quantity", "Silver", f"SELECT COUNT(*) FROM {S}.lineitem WHERE l_quantity IS NULL OR l_quantity<=0")
chk0(phase, "lineitem:null_extendedprice", "Silver", f"SELECT COUNT(*) FROM {S}.lineitem WHERE l_extendedprice IS NULL")
chk0(phase, "lineitem:null_discount", "Silver", f"SELECT COUNT(*) FROM {S}.lineitem WHERE l_discount IS NULL")
chk0(phase, "partsupp:null_supplycost", "Silver", f"SELECT COUNT(*) FROM {S}.partsupp WHERE ps_supplycost IS NULL")
print()

In [ ]:
phase = "reconciliation"
print(f"=== {phase.upper()} CHECKS ===")

# bronze.part = silver.part + silver.part_quarantine (content comparison)
part_cols = [r.col_name for r in spark.sql("DESCRIBE bda_bronze.part").collect()
             if r.col_name and not r.col_name.startswith("#") and r.col_name != "ingestion_timestamp"]
cl = ", ".join(part_cols)
chk0(phase, "part:bronze=(silver+quarantine)", "Silver",
    f"""SELECT COUNT(*) FROM (
        SELECT {cl} FROM bda_bronze.part
        EXCEPT ALL
        (SELECT {cl} FROM bda_silver.part
         UNION ALL
         SELECT {cl} FROM bda_silver.part_quarantine)
    )""")
chk0(phase, "part:(silver+quarantine)=bronze", "Silver",
    f"""SELECT COUNT(*) FROM (
        (SELECT {cl} FROM bda_silver.part
         UNION ALL
         SELECT {cl} FROM bda_silver.part_quarantine)
        EXCEPT ALL
        SELECT {cl} FROM bda_bronze.part
    )""")
print()

In [ ]:
phase = "gold"
G = "bda_gold"
S = "bda_silver"
B = "bda_bronze"
print(f"=== {phase.upper()} CHECKS ===")

# key equality both directions
chk0(phase, "sales_keys=silver_lineitem_keys", "Gold",
    f"""SELECT COUNT(*) FROM (
        SELECT l_orderkey, l_linenumber FROM {S}.lineitem
        EXCEPT ALL
        SELECT order_key, line_number FROM {G}.brand_sales)""")
chk0(phase, "silver_lineitem_keys=sales_keys", "Gold",
    f"""SELECT COUNT(*) FROM (
        SELECT order_key, line_number FROM {G}.brand_sales
        EXCEPT ALL
        SELECT l_orderkey, l_linenumber FROM {S}.lineitem)""")
chk0(phase, "sales_key_uniqueness", "Gold",
    f"SELECT COUNT(*) FROM (SELECT order_key, line_number FROM {G}.brand_sales GROUP BY order_key, line_number HAVING COUNT(*)>1)")

# revenue reconciliation (decimal, independently computed)
run_check(phase, "revenue: bronze=silver", "Gold", True,
    f"""SELECT ABS(
        (SELECT SUM(l_extendedprice*(1-l_discount)) FROM {B}.lineitem) -
        (SELECT SUM(l_extendedprice*(1-l_discount)) FROM {S}.lineitem)
    ) <= 0.01""")
run_check(phase, "revenue: silver=gold", "Gold", True,
    f"""SELECT ABS(
        (SELECT SUM(l_extendedprice*(1-l_discount)) FROM {S}.lineitem) -
        (SELECT SUM(net_revenue) FROM {G}.brand_sales)
    ) <= 0.01""")

# monthly revenue reconciliation
chk0(phase, "monthly_vs_detail_revenue", "Gold",
    f"""SELECT COUNT(*) FROM (
        SELECT m.month_start, m.category
        FROM (SELECT month_start, category, SUM(brand_revenue) AS mt
              FROM {G}.brand_category_monthly GROUP BY month_start, category) m
        FULL OUTER JOIN (SELECT month_start, category, SUM(net_revenue) AS dt
              FROM {G}.brand_sales GROUP BY month_start, category) d
          ON m.month_start=d.month_start AND m.category=d.category
        WHERE m.mt IS NULL OR (d.dt IS NULL AND m.mt <> 0)
           OR ABS(m.mt - COALESCE(d.dt, 0)) > 0.01
    )""")

# market share checks
chk0(phase, "shares_sum_to_1", "Gold",
    f"""SELECT COUNT(*) FROM (
        SELECT month_start, category, SUM(market_share) AS s
        FROM {G}.brand_category_monthly
        WHERE category_revenue > 0
        GROUP BY month_start, category
        HAVING ABS(SUM(market_share) - 1) > 0.01)""")
chk0(phase, "no_null_shares_when_positive", "Gold",
    f"SELECT COUNT(*) FROM {G}.brand_category_monthly WHERE category_revenue > 0 AND market_share IS NULL")
chk0(phase, "shares_in_0_1", "Gold",
    f"SELECT COUNT(*) FROM {G}.brand_category_monthly WHERE market_share < 0 OR market_share > 1")

# margin: gold vs independent silver reference
chk0(phase, "margin_matches_silver_ref", "Gold",
    f"""SELECT COUNT(*) FROM {G}.brand_sales bs
    JOIN {S}.part p ON bs.part_key=p.p_partkey
    JOIN {S}.partsupp ps ON bs.part_key=ps.ps_partkey AND bs.supplier_key=ps.ps_suppkey
    WHERE bs.list_unit_margin != (p.p_retailprice - ps.ps_supplycost)""")

# deviation-product key equality both directions
chk0(phase, "deviation_view=direct_ref", "Gold",
    f"""SELECT COUNT(*) FROM (
        SELECT part_key FROM {G}.price_deviation_products
        EXCEPT
        SELECT DISTINCT part_key FROM {G}.brand_sales WHERE price_deviation_ratio > 0.15)""")
chk0(phase, "direct_ref=deviation_view", "Gold",
    f"""SELECT COUNT(*) FROM (
        SELECT DISTINCT part_key FROM {G}.brand_sales WHERE price_deviation_ratio > 0.15
        EXCEPT
        SELECT part_key FROM {G}.price_deviation_products)""")

if validation_scope == "all":
    chk0(phase, "monitoring_grain_unique", "Gold",
        f"SELECT COUNT(*) FROM (SELECT month_start, brand, category FROM {G}.brand_monitoring GROUP BY month_start, brand, category HAVING COUNT(*)>1)")
    chk0(phase, "monitoring_consecutive_months", "Gold",
        f"""SELECT COUNT(*) FROM {G}.brand_monitoring m
        WHERE m.is_comparable_month AND (
            m.market_share IS NULL OR NOT m.is_complete_month OR NOT EXISTS (
                SELECT 1 FROM {G}.brand_monitoring p
                WHERE p.brand=m.brand AND p.category=m.category
                  AND p.month_start=ADD_MONTHS(m.month_start,-1)
                  AND p.is_complete_month AND p.market_share IS NOT NULL))""")
    chk0(phase, "share_change_identity", "Gold",
        f"SELECT COUNT(*) FROM {G}.brand_share_change_explanation WHERE identity_error_pp IS NULL OR identity_error_pp > 0.001")
print()

In [ ]:
phase = "persist"
print(f"=== {phase.upper()} ===")

# persist results to delta table
report_schema = "run_id string, checked_at timestamp, phase string, check_name string, layer string, expected_value string, actual_value string, passed boolean, is_critical boolean, details string"
df = spark.createDataFrame(results, report_schema)
df.write.mode("append").saveAsTable("bda_gold.validation_results")

# display summary
display(df.select("check_name", "layer", "passed", "is_critical", "details"))

passed = sum(1 for r in results if r.passed)
failed = sum(1 for r in results if not r.passed)
critical = sum(1 for r in results if not r.passed and r.is_critical)
print(f"\nTotal: {len(results)}  Passed: {passed}  Failed: {failed}  Critical failures: {critical}")

gold_version = spark.sql("DESCRIBE HISTORY bda_gold.brand_sales").select("version").orderBy("version", ascending=False).first()[0]
status = "failed" if critical else ("validated" if validation_scope == "all" else "gold_validated")
spark.sql(f"UPDATE bda_gold.run_audit SET validation_status='{status}' WHERE gold_version={gold_version}")

if critical > 0:
    print("\nCRITICAL FAILURES:")
    for r in results:
        if not r.passed and r.is_critical:
            print(f"  {r.check_name}: {r.details}")
    raise RuntimeError(f"{critical} critical validation failures. Pipeline stopped.")
print("All critical checks passed.")

In [ ]:
print("=== ERROR DEMONSTRATIONS ===")
print("Session temporary views only. Real tables are not modified.\n")

# derive keys from actual data
min_partkey = spark.sql("SELECT MIN(p_partkey) FROM bda_silver.part").collect()[0][0]
max_suppkey = spark.sql("SELECT MAX(s_suppkey) FROM bda_silver.supplier").collect()[0][0]
min_orderkey = spark.sql("SELECT MIN(l_orderkey) FROM bda_silver.lineitem").collect()[0][0]

# 1. empty brand
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW demo_part AS
    SELECT p_partkey, p_name, p_mfgr, '   ' AS p_brand, p_type, p_size,
           p_container, p_retailprice, p_comment
    FROM bda_silver.part WHERE p_partkey = {min_partkey}
""")
n = spark.sql("SELECT COUNT(*) FROM demo_part WHERE p_brand IS NULL OR TRIM(p_brand)=''").collect()[0][0]
assert n > 0, "empty brand not detected"
print("1. empty brand: detected")

# 2. duplicate partsupp pair (get real row, then duplicate it)
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW demo_partsupp AS
    WITH sample AS (
        SELECT ps_partkey, ps_suppkey FROM bda_silver.partsupp
        WHERE ps_partkey={min_partkey} ORDER BY ps_suppkey LIMIT 1
    )
    SELECT * FROM sample UNION ALL SELECT * FROM sample
""")
n = spark.sql("SELECT COUNT(*) FROM (SELECT ps_partkey, ps_suppkey FROM demo_partsupp GROUP BY ps_partkey, ps_suppkey HAVING COUNT(*)>1)").collect()[0][0]
assert n > 0, "duplicate partsupp not detected"
print("2. duplicate partsupp: detected")

# 3. missing product-supplier reference
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW demo_lineitem AS
    SELECT l_orderkey, l_partkey, {max_suppkey + 1} AS l_suppkey, l_linenumber,
           l_quantity, l_extendedprice, l_discount, l_tax, l_returnflag,
           l_linestatus, l_shipdate, l_commitdate, l_receiptdate,
           l_shipinstruct, l_shipmode, l_comment
    FROM bda_silver.lineitem WHERE l_orderkey = {min_orderkey} AND l_linenumber = 1
""")
n = spark.sql("""
    SELECT COUNT(*) FROM demo_lineitem l
    LEFT ANTI JOIN bda_silver.partsupp ps
    ON l.l_partkey = ps.ps_partkey AND l.l_suppkey = ps.ps_suppkey
""").collect()[0][0]
assert n > 0, "missing reference not detected"
print("3. missing reference: detected")

# 4. modified revenue
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW demo_lineitem_rev AS
    SELECT l_orderkey, l_partkey, l_suppkey, l_linenumber,
           l_quantity, 999999.99 AS l_extendedprice, l_discount, l_tax, l_returnflag,
           l_linestatus, l_shipdate, l_commitdate, l_receiptdate,
           l_shipinstruct, l_shipmode, l_comment
    FROM bda_silver.lineitem WHERE l_orderkey = {min_orderkey} AND l_linenumber = 1
""")
orig = spark.sql(f"SELECT SUM(l_extendedprice*(1-l_discount)) FROM bda_silver.lineitem WHERE l_orderkey={min_orderkey} AND l_linenumber=1").collect()[0][0]
demo = spark.sql("SELECT SUM(l_extendedprice*(1-l_discount)) FROM demo_lineitem_rev").collect()[0][0]
assert orig != demo, "modified revenue not detected"
print("4. modified revenue: detected")

# 5. null quantity
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW demo_lineitem_qty AS
    SELECT l_orderkey, l_partkey, l_suppkey, l_linenumber,
           CAST(NULL AS DECIMAL(18,2)) AS l_quantity, l_extendedprice, l_discount,
           l_tax, l_returnflag, l_linestatus, l_shipdate, l_commitdate,
           l_receiptdate, l_shipinstruct, l_shipmode, l_comment
    FROM bda_silver.lineitem WHERE l_orderkey = {min_orderkey} AND l_linenumber = 1
""")
n = spark.sql("SELECT COUNT(*) FROM demo_lineitem_qty WHERE l_quantity IS NULL OR l_quantity <= 0").collect()[0][0]
assert n > 0, "null quantity not detected"
print("5. null quantity: detected")

print("\nAll 5 error demonstrations confirmed.")